In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.append(
    str(PROJECT_ROOT)
)

In [2]:
from src.pricing.valuation import load_portfolio
portfolio = load_portfolio()
portfolio.columns.tolist()

['trade_id',
 'instrument_type',
 'underlying',
 'quantity',
 'strike',
 'maturity',
 'volatility',
 'dividend_yield',
 'coupon_rate',
 'coupon_frequency']

In [8]:
portfolio[
    portfolio["instrument_type"] == "FX_FORWARD"
]

,trade_id,instrument_type,underlying,quantity,strike,maturity,volatility,dividend_yield,coupon_rate,coupon_frequency
7,FX001,FX_FORWARD,EURUSD,2000000,1.1,2026-12-18,NaN,NaN,NaN,NaN


In [3]:
import pandas as pd

from src.pricing.valuation import (
    value_portfolio,
)
portfolio_risk = value_portfolio()

portfolio_risk

TypeError: forward_value() got an unexpected keyword argument 'maturity'

In [9]:
from src.fx_forward import forward_value
from src.pricing.valuation import value_portfolio
portfolio_risk = value_portfolio()
portfolio_risk

ImportError: cannot import name 'fx_forward_value' from 'src.fx_forward' (D:\market-risk-pnl-engine\src\fx_forward.py)

In [5]:
from src.fx_forward import forward_value

print(forward_value)

<function forward_value at 0x000002014E482770>


In [ ]:
options = portfolio_risk[
    portfolio_risk["instrument_type"].isin(
        [
            "EUROPEAN_CALL",
            "EUROPEAN_PUT",
        ]
    )
]

options[
    [
        "trade_id",
        "underlying",
        "market_value",
        "delta",
        "gamma",
        "vega",
        "rho",
        "theta",
    ]
]

In [ ]:
portfolio_risk[
    [
        "delta",
        "gamma",
        "vega",
        "rho",
        "theta",
        "ir_dv01",
    ]
].sum()

In [ ]:
equity_positions = portfolio_risk[
    portfolio_risk["instrument_type"]
    == "EQUITY"
]

equity_positions[
    [
        "trade_id",
        "underlying",
        "quantity",
        "spot",
        "market_value",
        "delta",
    ]
]

In [ ]:
import matplotlib.pyplot as plt

greek_values = portfolio_risk[
    [
        "delta",
        "gamma",
        "vega",
        "rho",
    ]
].sum()

plt.figure(figsize=(10, 5))

greek_values.plot(
    kind="bar"
)

plt.title(
    "Portfolio Risk Sensitivities"
)

plt.xlabel(
    "Risk Sensitivity"
)

plt.ylabel(
    "Exposure"
)

plt.tight_layout()

plt.show()

In [ ]:
portfolio_risk[
    [
        "trade_id",
        "delta",
    ]
].sort_values(
    "delta",
    ascending=False,
)

In [ ]:
portfolio_risk[
    [
        "trade_id",
        "gamma",
    ]
].sort_values(
    "gamma",
    ascending=False,
)

In [ ]:
portfolio_risk[
    [
        "trade_id",
        "vega",
    ]
].sort_values(
    "vega",
    ascending=False,
)

In [ ]:
portfolio_risk[
    [
        "trade_id",
        "ir_dv01",
    ]
].sort_values(
    "ir_dv01",
)

In [ ]:
from src.risk.sensitivity_pnl import (
    estimate_portfolio_shock_pnl,
)

shock = {
    "AAPL": -0.10 * portfolio_risk.loc[
        portfolio_risk["underlying"] == "AAPL",
        "spot",
    ].mean(),

    "SPY": -0.10 * portfolio_risk.loc[
        portfolio_risk["underlying"] == "SPY",
        "spot",
    ].mean(),

    "MSFT": -0.10 * portfolio_risk.loc[
        portfolio_risk["underlying"] == "MSFT",
        "spot",
    ].mean(),

    "EURUSD": 0.0,
    "US10Y": 0.0,
}

In [ ]:
shock_result = estimate_portfolio_shock_pnl(
    portfolio_risk,
    shock,
)

shock_result

In [ ]:
shock_result["estimated_pnl"].sum()